[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | Notebook

# How the Internet assigns and uses Autonomous Systems (ASes)

In [18]:
name = "Nathan Man"
date = "06/12/2026"

In [19]:
# Setup: shared utilities and tier definitions
import bz2
import urllib.request
import gzip
import io
import json
import zipfile
from contextlib import contextmanager
from pathlib import Path
from IPython.display import Markdown, display


@contextmanager
def open_safe(filename, encoding="utf-8"):
    """Open a file for reading, transparently handling .gz, .bz2, and .zip compression."""
    path = Path(filename)
    suffix = path.suffix.lower()
    if suffix == ".gz":
        with gzip.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".bz2":
        with bz2.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".zip":
        with zipfile.ZipFile(path) as zf:
            with zf.open(zf.namelist()[0]) as raw:
                yield io.TextIOWrapper(raw, encoding=encoding)
    else:
        with path.open(encoding=encoding) as f:
            yield f


TIERS = [
    ("edge",           1,     1),
    ("transit small",  2,     10),
    ("transit middle", 11,    1000),
    ("transit large",  1001,  10000),
    ("transit huge",   10001, -1),
]

---

## Task 2: ASN Classified by Customer Cone Size

Fill in the missing code in the cells below to produce **Table 1**.

### Data format

In `data/20260501.ppdc-ases.txt.bz2`, lines starting with `#` are comments.
All other lines start with an ASN followed by the ASNs in its customer cone.
The cone size for that ASN is the number of space-separated tokens on the line minus 1.

```
# This is a comment
# 23's customer cone size is 3 and includes (23, 4, 1)
23 23 4 1
# 1's customer cone size is 1 — it only includes itself
1 1
```

### Table 1 format

|           tier | range        | number of ASNs |   percentage |
| -------------: | ------------ | -------------: | -----------: |
|           edge | 1            |        [total] | [percentage] |
|  transit small | 2..10        |        [total] | [percentage] |
| transit middle | 11..1000     |        [total] | [percentage] |
|  transit large | 1001..10000  |        [total] | [percentage] |
|   transit huge | 10001..[max] |        [total] | [percentage] |

Replace `[max]` with the actual maximum cone size found in the data.
Percentages are formatted to one decimal place.

In [20]:
AS_CONE_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as-relationships/20260501.ppdc-ases.txt.bz2"
AS_CONE_PATH = Path("data/20260501.ppdc-ases.txt.bz2")

def classify(size: int) -> str:
    # YOUR CODE HERE
    # Use the TIERS list to determine which tier
    for label, low, high in TIERS:
        if high == -1 and size >= low:
            return label
        if low <= size <= high:
            return label
    return "unkown"
    # the given size falls into, and return the corresponding label.
    # Return "unknown" if the size does not fit into any tier.

print(f"Downloading {AS_CONE_URL} ...")
AS_CONE_PATH.parent.mkdir(parents=True, exist_ok=True)  # ensure 'data/' exists
urllib.request.urlretrieve(AS_CONE_URL, AS_CONE_PATH)
if not AS_CONE_PATH.exists(): 
    print (f"Unable to save file {AS_CONE_PATH}")
    exit() 

# Build a dict mapping each ASN to its customer cone size
asn_to_size = {} 
with open_safe(AS_CONE_PATH) as fin:
    for line in fin:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        # YOUR CODE HERE
        parts = line.split()
        asn = parts[0]
        asn_to_size[asn] = len(parts[1:])
        # count the number of ASN after the first
        # ASN on the line. 
        # 2 3 45 
        # asn_to_size["2"] = len(["3","45"])
print(f"Number of ASNs: {len(asn_to_size)}")


total = len(asn_to_size)
max_size = max(asn_to_size.values()) if asn_to_size else 0

tier_counts = {label: 0 for label, _, _ in TIERS}
for size in asn_to_size.values():
    tier_counts[classify(size)] += 1

header = "| tier | range | number of ASNs | percentage |"
sep    = "| ---: | ----- | ---: | ---: |"
table_2_rows = [header, sep]

for label, lo, hi in TIERS:
    # YOUR CODE HERE
    count = tier_counts[label]
    percentage = (count / total * 100) if total else 0
    rangeStr = ""
    if lo == hi:
        rangeStr = str(lo)
    elif hi == -1:
        rangeStr = str(lo) + "+"
    else:
        rangeStr = str(lo) + "-" + str(hi)
    table_2_rows.append(f"| {label} | {rangeStr} | {count} | {percentage:.1f}% |")
    
    # Compute the range string, count, and percentage 
    # for this tier, and append a row
table_2 = "\n".join(table_2_rows)

Number of ASNs: 79644


In [21]:
display(Markdown(table_2))
Path("tables").mkdir(exist_ok=True)
Path("tables/asn-cone-tiers.md").write_text(table_2, encoding="utf-8")

| tier | range | number of ASNs | percentage |
| ---: | ----- | ---: | ---: |
| edge | 1 | 67090 | 84.2% |
| transit small | 2-10 | 9977 | 12.5% |
| transit middle | 11-1000 | 2511 | 3.2% |
| transit large | 1001-10000 | 55 | 0.1% |
| transit huge | 10001+ | 11 | 0.0% |

270

In [22]:
max_size

56415

### Question 1

What percentage of ASNs are edge ASes (customer cone size of 1)? What does this suggest about the structure of the Internet?

84.2% of ASNs are edge ASes. This suggests that the majority of the Internet comprises of networks that don't provide transit for anyone else. This makes sense because it highlights how most networks on the Internet are consumers of the connections versus providers of it. 

### Question 2

How large is the maximum customer cone? What does this tell you about the most influential ASes on the Internet?

The maximum customer cone is 56415. This demonstrates how concentrated 

### Question 3

How do the proportions of edge, small transit, and large transit ASes compare? What does this distribution reveal about how ASes are organized hierarchically?

The proportions dramatically decrease as you go from edge to small transit to large transit ASes. The distribution is largely pyramidal with the edges acting as the base and huge transit at the tip. The number of ASes shrinks as you move up the heirarchy. 

---

## Task 3: How Are ASN Tiers Divided Across Countries?

Fill in the missing code in the cells below to produce **Table 2**.

- Use the tier classification from Task 2 (`classify()` and `TIERS`).
- Use `data/orgs.jsonl` to map each ASN to its organization's 2-letter country code.
  Each line is a JSON object with a `"country"` field and a `"members"` list of ASN strings.
- Find the top 4 countries by total ASN count across all tiers.
  All other countries map to **other**.
- The top-4 country codes become the dynamic column headers (e.g. `US`, `CN`).

### Table 2 format

| name           | 1st           | 2nd           | 3rd           | 4th           | other         |
| -------------- | ------------- | ------------- | ------------- | ------------- | ------------- |
| edge           | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit small  | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit middle | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit large  | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit huge   | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |

Replace `1st`/`2nd`/`3rd`/`4th` with the actual 2-letter country codes.
Each cell shows `total (X.X%)`, where `X.X%` is that country's share of the tier's total.

In [29]:
import json
AS2ORG_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as2org/as2org.jsonl"
AS2ORG_PATH = Path("data/as2org.jsonl")

print(f"Downloading {AS2ORG_URL} ...")
AS2ORG_PATH.parent.mkdir(parents=True, exist_ok=True)  # ensure 'data/' exists
urllib.request.urlretrieve(AS2ORG_URL, AS2ORG_PATH)
if not AS2ORG_PATH.exists(): 
    print (f"Unable to save file {AS2ORG_PATH}")
    exit() 
    
asn_to_country = {}
with open_safe(AS2ORG_PATH) as fin:
    for line in fin:    
        line = line.strip()
        if not line:
            continue
        # YOUR CODE HERE
        parsed = json.loads(line)
        country = parsed["country"]
        
        for member in parsed["members"]:
            asn_to_country[member] = country
        # Parse the JSON line, extract the ASN's in the members list
        # and country code, and add it to the asn_to_country dict.
print (f"number of ASNs with countries: {len(asn_to_country)}")

number of ASNs with countries: 120793


In [31]:
from collections import defaultdict

# Count ASNs per (tier, country) pair
tier_country_counts = defaultdict(lambda: defaultdict(int))
country_totals = defaultdict(int)

for asn, size in asn_to_size.items():
    tier = classify(size)
    country = asn_to_country.get(asn, "unknown")
    tier_country_counts[tier][country] += 1
    country_totals[country] += 1
    # YOUR CODE HERE: increment tier_country_counts[tier][country] and country_totals[country]

# Find top-4 countries by total ASN count
top4 = [c for c, _ in sorted(country_totals.items(), key=lambda x: -x[1])[:4]]
columns = top4 + ["other"]

# Build Markdown table
header = "| tier | " + " | ".join(columns) + " |"
sep    = "| --- | " + " | ".join(["---"] * len(columns)) + " |"
table_3_rows = [header, sep]

for label, lo, hi in TIERS:
    tier_total = sum(tier_country_counts[label].values())

    cells = []
    other_count = 0

    for country in top4:
        n = tier_country_counts[label][country]
        percentage = (n / tier_total * 100) if tier_total else 0
        cells.append(f"{n} ({percentage:.1f}%)")

    for country, count in tier_country_counts[label].items():
        if country not in top4:
            other_count += count
    other_percentage = (other_count / tier_total * 100) if tier_total else 0
    cells.append(f"{other_count} ({other_percentage:.1f}%)")
    table_3_rows.append(f"| {label} | " + " | ".join(cells) + " |")
# YOUR CODE HERE: for each (label, _, _) in TIERS, append a formatted row to table_3_rows
#   each cell: f"{n} ({pct:.1f}%)" where pct is the country's share of that tier's total
#   sum counts for countries not in top4 into "other"
table_3 = "\n".join(table_3_rows)

In [32]:
display(Markdown(table_3))

| tier | US | BR | RU | IN | other |
| --- | --- | --- | --- | --- | --- |
| edge | 16587 (24.7%) | 5943 (8.9%) | 4140 (6.2%) | 2475 (3.7%) | 37945 (56.6%) |
| transit small | 1582 (15.9%) | 1901 (19.1%) | 688 (6.9%) | 315 (3.2%) | 5491 (55.0%) |
| transit middle | 414 (16.5%) | 350 (13.9%) | 127 (5.1%) | 66 (2.6%) | 1554 (61.9%) |
| transit large | 10 (18.2%) | 7 (12.7%) | 8 (14.5%) | 3 (5.5%) | 27 (49.1%) |
| transit huge | 7 (63.6%) | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) | 4 (36.4%) |

### Question 4

Which countries have the most transit huge ASNs? What does this tell you about where Internet infrastructure is concentrated?

The US has the most transit huge ASNs (7 out of 11). This means that the Internet infrastructure is highly concentrated in the US.

### Question 5

What proportion of all ASNs fall in the "other" category? What does this suggest about the geographic distribution of the global Internet?

The "other" category is quite large (hovering around 50-60%) for most tiers except for transit huge which is around 36%. This means that the global internet is actually distributed quite diversely geographically even though the transit huge is quite concentrated. 

### Question 6

Do the same countries dominate across all AS tiers (edge, transit small, transit huge)? What patterns do you observe across the rows?

No the same countries do not dominate at every tier. The US is consistently strong at each level but does not dominate at the transit small level. For that level Brazil has the greatest amount of ASNs.